# 🌳 첫번째 함수 만들기

**날짜**: 2026-05-16  
**목표**: NDVI 코드를 재사용 가능한 함수로 만들기

## 🛠️ 만들 함수
1. `get_sentinel2_image()`: 영상 1장 가져오기
2. `calculate_ndvi()`: NDVI 계산

In [ ]:
# 라이브러리 가져오기
import ee
import geemap

# Earth Engine 초기화
ee.Initialize(project='knu-project-ndvi-2026')

print("준비 완료! 🛰️")

## 🛠️ 함수 1: get_sentinel2_image

Sentinel-2 영상 1장을 가져오는 함수.

### 📥 입력 (매개변수)
- `area`: 분석 영역 (ee.Geometry.Point 객체)
- `start_date`: 시작 날짜 (예: "2025-04-01")
- `end_date`: 끝 날짜 (예: "2025-04-20")
- `cloud_threshold`: 구름 비율 한계 (기본 10%)

### 📤 출력 (반환값)
- ee.Image: 조건에 맞는 영상 1장

### ⚙️ 동작
1. Sentinel-2 컬렉션 불러오기
2. 영역으로 필터링
3. 날짜로 필터링
4. 구름 비율로 필터링
5. 첫 영상 1장 반환

In [ ]:
# 함수 1: Sentinel-2 영상 가져오기
def get_sentinel2_image(area, start_date, end_date, cloud_threshold=10):
    """
    조건에 맞는 Sentinel-2 영상 1장을 가져온다.
    
    매개변수:
        area: 분석 영역 (ee.Geometry)
        start_date: 시작 날짜 (str)
        end_date: 끝 날짜 (str)
        cloud_threshold: 구름 비율 한계 (int, 기본 10)
    
    반환값:
        ee.Image: 영상 1장
    """
    image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(area)
             .filterDate(start_date, end_date)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
             .first())
    
    return image

print("함수 정의 완료! ✅")

In [ ]:
# 함수 2: NDVI 계산
def calculate_ndvi(image):
    """
    위성영상에서 NDVI를 계산한다.
    
    공식: (B8 - B4) / (B8 + B4)
    - B8: 근적외선 (식물이 강하게 반사)
    - B4: 빨강 (식물이 흡수)
    
    매개변수:
        image (ee.Image): Sentinel-2 영상
    
    반환값:
        ee.Image: NDVI 영상 (-1 ~ 1)
    """
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi


print("✅ 모든 함수 정의 완료!")
print("   - get_sentinel2_image()")
print("   - calculate_ndvi()")

In [ ]:
# 함수 테스트: 산불 전후 NDVI 비교

# 분석 영역: 대구 함지산
study_area = ee.Geometry.Point([128.5774, 35.9145])

# 산불 전 (2025년 4월 초)
image_before = get_sentinel2_image(study_area, '2025-04-01', '2025-04-20')
ndvi_before = calculate_ndvi(image_before)

# 산불 후 (2025년 5월)
image_after = get_sentinel2_image(study_area, '2025-04-29', '2025-05-15')
ndvi_after = calculate_ndvi(image_after)

print("✅ 산불 전후 NDVI 계산 완료!")

In [ ]:
# NDVI 비교 지도 만들기

# 시각화 설정
ndvi_vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# 지도 생성
Map = geemap.Map(center=[35.9145, 128.5774], zoom=12)

# 두 시기 NDVI 레이어 추가
Map.addLayer(ndvi_before, ndvi_vis_params, '산불 전 NDVI (2025-04)')
Map.addLayer(ndvi_after, ndvi_vis_params, '산불 후 NDVI (2025-05)')

Map